# SETTING UP ACCESS TO GITHUB (PRESERVED STATE) FOR EVERY NEW RUNTIME

In [ ]:
from google.colab import userdata
import os

git_token = userdata.get('GIT_TOKEN')
git_username = "TalhaShoyo10"
repo_name = "CS-6304_PA0_28100131"
repo_url = f"https://{git_token}@github.com/{git_username}/{repo_name}.git"



if not os.path.exists(repo_name):
  !git clone {repo_url}
  print("Repo did not exist in local file system, cloned from github to initiate work on task.")
else:
  !git -C {repo_name} pull
  print("Repo already existed in local file system, pulled from github to catch up on any remote changes.")

Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# VARIATIONAL AUTOENCODER

Loading MNIST

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Number of training examples: {len(train_dataset)}")
print(f"Number of testing examples: {len(test_dataset)}")
print(f"Number of training batches: {len(train_loader)}")

# Building a Basic VAE

Defining the Encoder, Decoder, and Reparameterization Trick

In [ ]:
LATENT_DIM = 2  
HIDDEN_DIM = 400
INPUT_DIM = 28 * 28

class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()

        #Encoder: x -> hidden -> (mu, logvar)
        self.encoder_fc1 = nn.Linear(input_dim, hidden_dim)
        self.encoder_mu = nn.Linear(hidden_dim, latent_dim)
        self.encoder_logvar = nn.Linear(hidden_dim, latent_dim)

        #Decoder: z -> hidden -> reconstructed pixel logits
        self.decoder_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.encoder_fc1(x))
        mu = self.encoder_mu(h)
        logvar = self.encoder_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        #z = mu + sigma * epsilon, epsilon ~ N(0, I)
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + std * epsilon
        return z

    def decode(self, z):
        h = F.relu(self.decoder_fc1(z))
        pixel_logits = self.decoder_out(h)
        return pixel_logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        pixel_logits = self.decode(z)
        return pixel_logits, mu, logvar


model = VAE().to(device)
print(model)

trainable_paras = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_paras}")

Defining the VAE Loss (Negative ELBO)

In [ ]:
def vae_loss(pixel_logits, x, mu, logvar):
    #Reconstruction term: -E_q[log p(x|z)], using a Bernoulli output distribution over pixels
    recon_loss = F.binary_cross_entropy_with_logits(pixel_logits, x, reduction="sum")

    #KL term: closed-form KL divergence between N(mu, sigma^2) and N(0, I)
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + kl_div
    return total_loss, recon_loss, kl_div

Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_recon, total_kl = 0.0, 0.0, 0.0

    for images, _ in loader:
        images = images.view(images.size(0), -1).to(device)

        optimizer.zero_grad()
        pixel_logits, mu, logvar = model(images)
        loss, recon_loss, kl_div = vae_loss(pixel_logits, images, mu, logvar)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_div.item()

    n = len(loader.dataset)
    return total_loss / n, total_recon / n, total_kl / n


optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 10

history = {"train_loss": [], "recon_loss": [], "kl_loss": []}

for epoch in range(EPOCHS):
    avg_loss, avg_recon, avg_kl = train_epoch(model, train_loader, optimizer, device)
    history["train_loss"].append(avg_loss)
    history["recon_loss"].append(avg_recon)
    history["kl_loss"].append(avg_kl)
    print(f"Epoch {epoch+1}/{EPOCHS} - ELBO Loss: {avg_loss:.4f} (Recon: {avg_recon:.4f}, KL: {avg_kl:.4f})")

Plotting the ELBO Loss Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history["train_loss"], label="Total ELBO Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (per example)")
axes[0].set_title("VAE Training Loss")
axes[0].legend()

axes[1].plot(history["recon_loss"], label="Reconstruction Loss")
axes[1].plot(history["kl_loss"], label="KL Divergence")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss (per example)")
axes[1].set_title("Reconstruction vs. KL Terms")
axes[1].legend()

plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task4_vae/training_loss.png", dpi=150)
plt.show()

# Analyzing VAE Performance

(a) Latent Space Visualization

In [ ]:
model.eval()

all_mu = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1).to(device)
        mu, logvar = model.encode(images)
        all_mu.append(mu.cpu())
        all_labels.append(labels)

all_mu = torch.cat(all_mu, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

plt.figure(figsize=(8, 7))
scatter = plt.scatter(all_mu[:, 0], all_mu[:, 1], c=all_labels, cmap="tab10", s=5, alpha=0.6)
plt.colorbar(scatter, ticks=range(10), label="Digit label")
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.title("VAE Latent Space (MNIST Test Set, mu_phi(x))")
plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task4_vae/latent_space.png", dpi=150)
plt.show()

(b) Reconstruction Quality

In [ ]:
NUM_RECON_SAMPLES = 8

sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images[:NUM_RECON_SAMPLES]
sample_labels = sample_labels[:NUM_RECON_SAMPLES]

flat_images = sample_images.view(NUM_RECON_SAMPLES, -1).to(device)

with torch.no_grad():
    pixel_logits, mu, logvar = model(flat_images)
    reconstructions = torch.sigmoid(pixel_logits).view(NUM_RECON_SAMPLES, 1, 28, 28).cpu()

fig, axes = plt.subplots(2, NUM_RECON_SAMPLES, figsize=(2 * NUM_RECON_SAMPLES, 4))

for i in range(NUM_RECON_SAMPLES):
    axes[0, i].imshow(sample_images[i].squeeze(), cmap="gray")
    axes[0, i].set_title(f"orig: {sample_labels[i].item()}")
    axes[0, i].axis("off")

    axes[1, i].imshow(reconstructions[i].squeeze(), cmap="gray")
    axes[1, i].set_title("recon")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task4_vae/reconstructions.png", dpi=150)
plt.show()

(c) Generation of New Samples

In [ ]:
NUM_GENERATED = 10

with torch.no_grad():
    z_random = torch.randn(NUM_GENERATED, LATENT_DIM).to(device)
    generated_logits = model.decode(z_random)
    generated_images = torch.sigmoid(generated_logits).view(NUM_GENERATED, 1, 28, 28).cpu()

fig, axes = plt.subplots(1, NUM_GENERATED, figsize=(2 * NUM_GENERATED, 2))

for i in range(NUM_GENERATED):
    axes[i].imshow(generated_images[i].squeeze(), cmap="gray")
    axes[i].axis("off")

plt.suptitle("Samples generated from z ~ N(0, I)")
plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task4_vae/generated_samples.png", dpi=150)
plt.show()

Discussion points to fill in after reviewing the visualizations above:

- Latent space: Do the clusters for each digit make sense? Is there a continuous progression between visually similar digits (e.g. 4/9, 3/8)?
- Reconstructions: Are particular digits reconstructed better or worse than others? Any notable blurriness?
- Generations: Do the randomly sampled digits look like valid handwritten digits, and do they reflect the diversity of the dataset?

## Comparing with Doersch's VAE Implementation

Discussion points to fill in after reviewing Doersch's tutorial and reference implementation:

- Architecture: How does the MLP used here (784 -> 400 -> latent_dim, and back) compare to Doersch's architecture in terms of layers/hidden units?
- Output Distribution and Loss: Doersch treats pixel intensities as probabilities and uses a sigmoid cross-entropy loss (Bernoulli output), matching the approach used here (`binary_cross_entropy_with_logits`). Confirm this matches and note any differences.
- Latent Dimensionality: This implementation fixes `LATENT_DIM = 2` for visualization purposes. Consider experimenting with a larger latent dimension (e.g. 10 or 20) and comparing reconstruction quality / training difficulty against Doersch's observation that VAEs are largely insensitive to z's dimensionality outside of extreme values.
- Results Comparison: Qualitatively compare the generated digits and reconstructions above with Doersch's reported samples. Note any similar "in-between"/ambiguous digit failure cases.

Saving Results

In [ ]:
vae_results = {
    "latent_dim": LATENT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-3,
    "history": history,
    "final_train_loss": history["train_loss"][-1],
}

with open("CS-6304_PA0_28100131/results/task4_vae/vae_results.json", "w") as f:
    json.dump(vae_results, f, indent=2)

torch.save(model.state_dict(), "CS-6304_PA0_28100131/results/task4_vae/vae_model_weights.pt")

print("Results saved successfully !!")

Checking for Saved Files

In [ ]:
print(os.listdir("CS-6304_PA0_28100131/results/task4_vae"))

Git Configuration

In [ ]:
!git config --global user.email "thebenbat5@gmail.com"
!git config --global user.name "TalhaShoyo10"

Commiting work to Github

In [ ]:
commit_message = "Task 4 - VAE implementation, training, and analysis"
!git -C {repo_name} add -A
!git -C {repo_name} commit -m "{commit_message}"
!git -C {repo_name} push https://{git_token}@github.com/{git_username}/{repo_name}.git main